# Notebook 03: Global Sensitivity Analysis (Morris Method)

**Project:** Brent Crude Oil Prices and Renewable Energy Investment  
**Date:** 2026-04-06  

This notebook runs a Morris method global sensitivity analysis (GSA) on the
CES investment model to identify which parameters most influence the 2030
renewable investment output.

Methodology adapted from Usher et al. (2023) esom_gsa (MIT licence).

## Parameters Screened

| Parameter | Range | Source |
|---|---|---|
| sigma | [1.0, 3.0] | Papageorgiou et al. 2017 |
| alpha | [0.15, 0.60] | Calibrated to 2024 share |
| oil_price_elasticity | [0.05, 0.35] | Mukhtarov et al. 2024 CI |
| oil_price_2025 | [30, 110] | IEA/IMF scenario bounds |
| oil_price_2030 | [25, 130] | Boer et al. / IMF WP/23/160 |
| discount_rate | [0.05, 0.15] | WACC estimates |
| renew_capex_decline | [0.0, 0.50] | IEA learning curve bounds |
| invest_baseline_bn_usd | [600, 1200] | IEA/IRENA uncertainty range |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from ces_model.sensitivity.problem import build_problem, get_baselines
from ces_model.sensitivity.sampler import sample_morris
from ces_model.sensitivity.runner import run_sensitivity
from ces_model.sensitivity.analyser import analyse_morris, save_results

import os
os.makedirs("figures", exist_ok=True)

print("ces_model sensitivity module loaded.")

## 1. Build the SALib Problem

In [ ]:
problem = build_problem()

print(f"Number of parameters: {problem['num_vars']}")
print(f"Groups: {sorted(set(problem['groups']))}")
print()
for name, bounds, group in zip(problem["names"], problem["bounds"], problem["groups"]):
    print(f"  {name:<30} [{bounds[0]:6.2f}, {bounds[1]:6.2f}]   group={group}")

## 2. Generate Morris Sample

In [ ]:
NUM_TRAJECTORIES = 10
NUM_LEVELS = 4
SEED = 42

sample = sample_morris(
    problem,
    num_trajectories=NUM_TRAJECTORIES,
    num_levels=NUM_LEVELS,
    seed=SEED,
)

n_groups = len(set(problem["groups"]))
expected_rows = (n_groups + 1) * NUM_TRAJECTORIES
print(f"Sample shape: {sample.shape}")
print(f"Expected rows: ({n_groups} groups + 1) x {NUM_TRAJECTORIES} = {expected_rows}")

## 3. Run the CES Model for Each Sample Row

In [ ]:
# n_workers=1 for notebook safety; set higher for batch runs
Y = run_sensitivity(sample, problem, n_workers=1)

print(f"Y shape: {Y.shape}")
print(f"Y stats: min={Y.min():.1f}  mean={Y.mean():.1f}  max={Y.max():.1f}  (USD bn)")
print(f"\nOutput = terminal 2030 renewable investment (USD bn)")

## 4. Morris Analysis

In [ ]:
sensitivity_result = analyse_morris(problem, sample, Y, num_levels=NUM_LEVELS)

results_df = sensitivity_result.to_dataframe()
print("Morris Sensitivity Indices (ranked by mu_star):")
print(results_df.to_string(index=False))

In [ ]:
# Bar chart: mu_star with sigma error bars
fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(results_df))
ax.barh(
    y_pos,
    results_df["mu_star"],
    xerr=results_df["sigma"],
    align="center",
    color="steelblue",
    ecolor="black",
    capsize=4,
)
ax.set_yticks(y_pos)
ax.set_yticklabels(results_df["parameter"])
ax.set_xlabel("Morris mu* (mean absolute elementary effect, USD bn)")
ax.set_title(
    "Morris GSA: Parameter Influence on 2030 Renewable Investment"
)
ax.invert_yaxis()
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("figures/03a_morris_mu_star.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. mu vs sigma Plot

A scatter of mu (signed mean effect) vs sigma (standard deviation of effects)
identifies:
- High mu, low sigma: linear monotonic effect
- High sigma: nonlinear or interaction effects

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Guidelines: sigma = mu_star (equal-effect boundary)
max_val = max(results_df["mu_star"].max(), results_df["sigma"].max()) * 1.15
diag = np.linspace(0, max_val, 100)
ax.plot(diag, diag, "--", color="gray", linewidth=0.8, label="sigma = mu*")
ax.plot(diag, 0.5 * diag, ":", color="gray", linewidth=0.8, label="sigma = 0.5 mu*")

ax.scatter(
    results_df["mu_star"],
    results_df["sigma"],
    s=80,
    color="steelblue",
    zorder=5,
)
for _, row in results_df.iterrows():
    ax.annotate(
        row["parameter"],
        (row["mu_star"], row["sigma"]),
        textcoords="offset points",
        xytext=(5, 3),
        fontsize=8,
    )

ax.set_xlabel("mu* (mean absolute elementary effect)")
ax.set_ylabel("sigma (standard deviation of effects)")
ax.set_title("Morris GSA: mu* vs sigma -- Nonlinearity Identification")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/03b_morris_mu_sigma.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Output Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(Y, bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(Y.mean(), color="darkorange", linewidth=1.5, label=f"Mean = {Y.mean():.1f}")
axes[0].axvline(np.percentile(Y, 5), color="firebrick", linewidth=1.2,
                linestyle="--", label=f"P5 = {np.percentile(Y, 5):.1f}")
axes[0].axvline(np.percentile(Y, 95), color="forestgreen", linewidth=1.2,
                linestyle="--", label=f"P95 = {np.percentile(Y, 95):.1f}")
axes[0].set_xlabel("2030 Renewable Investment (USD bn)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Model Output (Morris Sample)")
axes[0].legend(fontsize=8)

sorted_y = np.sort(Y)
cdf = np.arange(1, len(sorted_y) + 1) / len(sorted_y)
axes[1].plot(sorted_y, cdf, color="steelblue", linewidth=2)
axes[1].axhline(0.05, color="firebrick", linewidth=0.8, linestyle="--")
axes[1].axhline(0.95, color="forestgreen", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("2030 Renewable Investment (USD bn)")
axes[1].set_ylabel("Cumulative Probability")
axes[1].set_title("Empirical CDF of Model Output")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("figures/03c_output_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOutput statistics (Morris sample, n={len(Y)}):")
print(f"  Min:  ${Y.min():.1f} bn")
print(f"  P5:   ${np.percentile(Y, 5):.1f} bn")
print(f"  Mean: ${Y.mean():.1f} bn")
print(f"  P95:  ${np.percentile(Y, 95):.1f} bn")
print(f"  Max:  ${Y.max():.1f} bn")

## 7. Key Findings

This cell summarises the top influential parameters and links the sensitivity
findings back to the scenario design.

In [ ]:
top5 = sensitivity_result.top_parameters(5)
baselines = get_baselines()

print("=" * 55)
print("MORRIS GSA: KEY FINDINGS")
print("=" * 55)
print(f"\nTop 5 influential parameters (by mu*):")
for rank, param in enumerate(top5, 1):
    row = results_df[results_df["parameter"] == param].iloc[0]
    baseline = baselines.get(param, float("nan"))
    print(
        f"  {rank}. {param:<30} "
        f"mu*={row['mu_star']:.2f}  "
        f"sigma={row['sigma']:.2f}  "
        f"baseline={baseline}"
    )

print()
print("Interpretation:")
print("  Parameters with high mu* most influence 2030 renewable investment.")
print("  Parameters with high sigma/mu* ratio have nonlinear or interaction effects.")
print("  Oil price variables and sigma are expected to rank highly given sigma=1.8.")

In [ ]:
# Save results to CSV
paths = save_results(sensitivity_result, output_dir="figures", prefix="morris_gsa")
print(f"Saved: {paths['csv']}")
print(f"Saved: {paths['png']}")